# Step 14.2 — LayerNorm → RMSNorm Historical Path：为什么 Transformer 需要控制 Activation Scale？

Step 14.1 已经有：

$$
x_{new}=x+F(x).
$$

Residual connection 让 representation 可以持续累积 learned updates。

但层数变深以后，一个新的问题变得重要：

> **不同 token / layer 的 feature magnitudes 与 offsets 可能持续变化，后续 sublayer 每次看到的数值尺度会变得更难控制。**

Normalization 的目标不是“让所有信息一样”，而是给每个 token 的 feature vector 一个更可控的数值参考尺度。

历史上 Transformer 最经典的是 **LayerNorm（Layer Normalization）**；后来许多现代 LLM 采用更简化的 **RMSNorm（Root Mean Square Normalization）**。

本节只讲 LayerNorm → RMSNorm 的数学与高层历史关系；不进入 Step 14.3 的 Pre-Norm / Post-Norm 放置问题。


## 1. Transformer 里的 Normalization 到底沿哪个 Axis？

输入 representation：

$$
x\in\mathbb R^{B\times T\times C}.
$$

LayerNorm / RMSNorm 在这里都把每个 token 的：

$$
x_{b,t,:}\in\mathbb R^C
$$

单独拿出来做 normalization。

也就是说：

```text
固定 b
固定 t
只在最后一个 feature axis C 上计算统计量
```

不是：

- 跨 batch 求均值；
- 跨 sequence positions 求均值；
- 把不同 token 混在一起。

因此 normalization 前后 shape 都保持：

$$
\boxed{(B,T,C)\rightarrow(B,T,C)}.
$$


## 2. LayerNorm：先 Center，再 Scale

对一个 token vector：

$$
x=[x_1,\ldots,x_C].
$$

先计算 feature mean：

$$
\mu
=
\frac{1}{C}\sum_{i=1}^{C}x_i.
$$

再计算 population variance：

$$
\sigma^2
=
\frac{1}{C}\sum_{i=1}^{C}(x_i-\mu)^2.
$$

归一化：

$$
\hat{x}_i
=
\frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}}.
$$

然后再使用 learned per-feature parameters：

$$
\boxed{
y_i
=
\gamma_i\hat{x}_i+\beta_i
}.
$$

这里：

```text
epsilon (ε) -> numerical stability constant
gamma (γ)   -> learned per-feature scale
beta  (β)   -> learned per-feature shift
```

LayerNorm 的关键特征是：

> **它会减去 mean，因此同时处理 offset 与 scale。**


## 3. 手算 LayerNorm：`x=[1,2,3,4]`

暂时忽略极小的 epsilon，并先设：

$$
\gamma=[1,1,1,1],
\qquad
\beta=[0,0,0,0].
$$

输入：

$$
x=[1,2,3,4].
$$

Mean：

$$
\mu=2.5.
$$

Centered values：

$$
[-1.5,-0.5,0.5,1.5].
$$

Variance：

$$
\sigma^2
=
\frac{2.25+0.25+0.25+2.25}{4}
=
1.25.
$$

Standard deviation：

$$
\sqrt{1.25}\approx1.1180.
$$

所以 LayerNorm normalized output 约为：

$$
\boxed{
[-1.3416,-0.4472,0.4472,1.3416]
}.
$$

归一化后的 feature mean 约为 0，feature variance 约为 1；之后 learned gamma / beta 可以重新调节各 feature 的 scale 与 offset。


In [ ]:
import mlx.core as mx
import mlx.nn as nn

x = mx.array([[1.0, 2.0, 3.0, 4.0]])  # (1,C)

mu = mx.mean(x, axis=-1, keepdims=True)
variance = mx.mean((x - mu) ** 2, axis=-1, keepdims=True)

eps = 1e-5
manual_ln = (x - mu) / mx.sqrt(variance + eps)

layer_norm = nn.LayerNorm(
    dims=4,
    eps=eps,
    affine=True,
    bias=True,
)

print('mean:', mu)
print('variance:', variance)
print('manual LayerNorm before learned affine:', manual_ln)
print('MLX LayerNorm:', layer_norm(x))

# MLX initializes gamma(weight)=1 and beta(bias)=0, so initially the two
# expressions are expected to match apart from floating-point rounding.
# This master notebook remains unexecuted.


## 4. `epsilon` 为什么一定要有？

假设一个 token 的所有 features 完全相同：

$$
x=[3,3,3,3].
$$

LayerNorm 的 variance：

$$
\sigma^2=0.
$$

如果没有 epsilon：

$$
\frac{x_i-\mu}{\sqrt{0}}
$$

会遇到除零问题。

因此使用：

$$
\sqrt{\sigma^2+\epsilon}.
$$

`epsilon` 是一个很小的正数，用于保证 denominator 不会变成 0，并改善数值稳定性。

它不是 learned parameter，也不是为了给输出“加一点噪声”。

MLX 0.32.2 的 `LayerNorm` 与 `RMSNorm` 默认：

```text
eps = 1e-5
```


## 5. RMSNorm：保留 Scale Normalization，去掉 Mean-Centering

RMSNorm 的核心统计量不是 variance，而是：

$$
\operatorname{RMS}(x)
=
\sqrt{
\frac{1}{C}
\sum_{i=1}^{C}x_i^2
+\epsilon
}.
$$

然后：

$$
\boxed{
y_i
=
\gamma_i
\frac{x_i}{\operatorname{RMS}(x)}
}.
$$

和 LayerNorm 相比，RMSNorm：

```text
不计算 mean μ
不做 x - μ
直接用 root mean square 控制向量尺度
保留 learned per-feature scale gamma
```

在 MLX 0.32.2 的 `nn.RMSNorm` 中没有 beta shift parameter。

所以 RMSNorm 的核心 simplification 是：

> **保留 re-scaling，去掉 re-centering。**


## 6. 同一个 `x=[1,2,3,4]` 看 RMSNorm

仍先设：

$$
\gamma=[1,1,1,1]
$$

并暂时忽略很小的 epsilon。

Mean square：

$$
\frac{1^2+2^2+3^2+4^2}{4}
=
\frac{30}{4}
=
7.5.
$$

RMS：

$$
\sqrt{7.5}\approx2.7386.
$$

所以：

$$
\frac{x}{\operatorname{RMS}(x)}
\approx
\boxed{
[0.3651,0.7303,1.0954,1.4606]
}.
$$

注意这个结果的 feature mean 并不是 0。

这正是 RMSNorm 与 LayerNorm 最直观的区别：

> RMSNorm 控制 overall magnitude，但不把 feature vector 强制 center 到 zero mean。


In [ ]:
mean_square = mx.mean(x ** 2, axis=-1, keepdims=True)
rms = mx.sqrt(mean_square + eps)
manual_rms = x / rms

rms_norm = nn.RMSNorm(
    dims=4,
    eps=eps,
)

print('mean square:', mean_square)
print('RMS:', rms)
print('manual RMSNorm before learned gamma:', manual_rms)
print('MLX RMSNorm:', rms_norm(x))

# MLX RMSNorm initializes gamma(weight)=1, so initial output is expected
# to match the manual expression apart from floating-point rounding.
# The master notebook remains unexecuted.


## 7. 为什么 Normalize 完还要 Learned `gamma`？

如果 normalization 永远强制所有 features 保持完全固定 scale，会限制模型的表示自由度。

因此 LayerNorm / RMSNorm 都使用 learned per-feature gain：

$$
\gamma\in\mathbb R^C.
$$

例如 normalized vector：

$$
z=[1,-1,0.5,-0.5]
$$

若学习到：

$$
\gamma=[2,0.5,1,3]
$$

则输出变成：

$$
[2,-0.5,0.5,-1.5].
$$

也就是说：

> normalization 先建立一个受控的数值参考尺度，gamma 再让模型学习每个 feature 最合适的幅度。

LayerNorm 通常还可以有 beta；RMSNorm 的常见形式只保留 gamma。


## 8. Historical Path：为什么从 LayerNorm 走向 RMSNorm？

### LayerNorm

Layer Normalization 在 2016 年提出，后来成为原始 Transformer 及大量序列模型中的核心 normalization 方法。

它同时提供：

```text
re-centering : subtract mean
re-scaling   : divide by standard deviation
learned affine recovery : gamma / beta
```

### RMSNorm

RMSNorm 在 2019 年提出，思路是：

> 如果模型主要需要控制 hidden representation 的尺度，那么是否可以省掉 mean-centering，只保留基于 RMS 的 re-scaling？

因此 RMSNorm 去掉了 mean calculation / subtraction，保留 root-mean-square scaling 与 learned gamma。

后来许多现代 LLM architectures 采用 RMSNorm，但这不意味着：

- RMSNorm 在任何任务上都必然优于 LayerNorm；
- LayerNorm 已经过时或错误；
- 二者只是名字不同、数学完全一样。

准确说法是：

> **RMSNorm 是对 LayerNorm normalization idea 的一种简化路径：放弃 re-centering，保留 re-scaling。**


## 9. LayerNorm vs RMSNorm：现在应该怎样准确区分？

| 属性 | LayerNorm | RMSNorm |
|---|---|---|
| 每个 token 独立处理 | 是 | 是 |
| 沿 feature axis C | 是 | 是 |
| subtract mean | 是 | 否 |
| 使用 variance / std | 是 | 否 |
| 使用 RMS | 间接不是核心定义 | 是 |
| learned gamma | 通常有 | 有 |
| learned beta | 常见实现可有 | 常见 RMSNorm 无 |
| shape `(B,T,C)` 保持 | 是 | 是 |

一个有用的高层记忆：

$$
\boxed{
\text{LayerNorm}
=
\text{center}+\text{scale}
}
$$

而：

$$
\boxed{
\text{RMSNorm}
=
\text{scale only}
}
$$

这里的“scale only”指 normalization statistic 不做 mean-centering，不代表 RMSNorm 没有 learned gamma。


## 10. MLX 0.32.2 API

项目固定：

```text
mlx==0.32.2
```

LayerNorm：

```python
nn.LayerNorm(
    dims=C,
    eps=1e-5,
    affine=True,
    bias=True,
)
```

其 learned parameters 默认初始化：

```text
gamma / weight = 1
beta  / bias   = 0
```

RMSNorm：

```python
nn.RMSNorm(
    dims=C,
    eps=1e-5,
)
```

其 learned parameter：

```text
gamma / weight = 1
```

MLX 0.32.2 的 RMSNorm mean-square accumulation 使用 float32 precision。

这一节只认识数学与 API；normalization 到底放在 residual branch 的前面还是后面，是 Step 14.3。


## 11. Step 14.2 材料边界

- Transformer 的 LayerNorm / RMSNorm 都按 token 独立地沿最后一个 feature axis C 做 normalization，不跨 batch 或 sequence positions 混合统计量。
- LayerNorm：`(x-mu)/sqrt(var+eps)`，再乘 learned gamma，并可加 beta；它同时 re-center 与 re-scale。
- RMSNorm：`x/sqrt(mean(x^2)+eps)` 再乘 learned gamma；它不做 mean-centering。
- epsilon 是固定的 numerical-stability constant，不是 learned parameter。
- gamma 是 learned per-feature scale，用来在 normalization 后恢复/调整 feature-specific magnitude。
- MLX 0.32.2：`nn.LayerNorm(dims, eps=1e-5, affine=True, bias=True)`；`nn.RMSNorm(dims, eps=1e-5)`，后者没有 beta。
- RMSNorm 可以理解为从 LayerNorm 思路中省去 re-centering、保留 re-scaling 的简化路径；不应说它在所有场景必然优于 LayerNorm。
- 本轮只准备 Step 14.2；不进入 Step 14.3，不讨论 normalization 放在 residual branch 前还是后，也不展开 Pre-Norm / Post-Norm。
